# Feature maps: lift your data
**Machine Learning · Week 4 (Kernels and support vector machines) · Day 1**

Lesson page: https://sara-aminpour.com/tutorials/ml/w4d1.html

A ring around a dot can't be split by a straight line. Add one new feature,
$z = x_1^2 + x_2^2$, and a flat plane splits it perfectly.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_circles
from sklearn.linear_model import LogisticRegression

X, y = make_circles(n_samples=300, noise=0.08, factor=0.45, random_state=0)
flat = LogisticRegression().fit(X, y)
print("straight line in 2D, accuracy:", round(flat.score(X, y), 3))

About 50%: no better than guessing. Now the **feature map** $\varphi(x_1, x_2) = (x_1,\; x_2,\; x_1^2 + x_2^2)$.

In [ ]:
def lift(X):
    return np.c_[X, (X ** 2).sum(axis=1)]      # add a third column: the squared distance from the center

Z = lift(X)
lifted = LogisticRegression().fit(Z, y)
print("shape before:", X.shape, "  after:", Z.shape)
print("flat plane in 3D, accuracy:", round(lifted.score(Z, y), 3))

In [ ]:
colors = np.where(y == 0, "#ff8a50", "#5577f5")
w1, w2, w3 = lifted.coef_[0]
b = lifted.intercept_[0]

fig = plt.figure(figsize=(6, 4.6))
ax = fig.add_subplot(projection="3d")
ax.scatter(Z[:, 0], Z[:, 1], Z[:, 2], c=colors, s=12, depthshade=False)
gx, gy = np.meshgrid(np.linspace(-1.2, 1.2, 10), np.linspace(-1.2, 1.2, 10))
gz = -(w1 * gx + w2 * gy + b) / w3                  # the plane w·z + b = 0, solved for the third coordinate
ax.plot_surface(gx, gy, gz, color="#b9a7ff", alpha=0.35)
ax.set_xlabel("x1")
ax.set_ylabel("x2")
ax.set_zlabel("x1² + x2²")
ax.view_init(elev=16, azim=-58)
ax.set_title("Lifted into 3D: a flat plane separates the classes")
plt.show()

## Back down in 2D
The flat plane in 3D becomes a **circle** in the original 2D picture.

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

apricot_blue = LinearSegmentedColormap.from_list("apricot_blue", ["#ff8a50", "#ffffff", "#5577f5"])
xx, yy = np.meshgrid(np.linspace(-1.5, 1.5, 250), np.linspace(-1.5, 1.5, 250))
p = lifted.predict_proba(lift(np.c_[xx.ravel(), yy.ravel()]))[:, 1].reshape(xx.shape)

fig, ax = plt.subplots(figsize=(4.6, 4.2), layout="constrained")
ax.contourf(xx, yy, p, levels=20, cmap=apricot_blue, alpha=0.5)
ax.contour(xx, yy, p, levels=[0.5], colors="#2c2842", linewidths=2)
ax.scatter(X[:, 0], X[:, 1], c=colors, s=12)
ax.set_aspect("equal")
ax.set_title("A linear model on lifted features")
plt.show()

## Let scikit-learn build the features
`PolynomialFeatures(2)` makes every product up to degree 2. A linear model on top can draw circles,
ellipses and parabolas.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures

print("degree-2 features:", PolynomialFeatures(2).fit(X).get_feature_names_out(["x1", "x2"]))
auto = make_pipeline(PolynomialFeatures(2), LogisticRegression()).fit(X, y)
print("accuracy:", round(auto.score(X, y), 3))
for d in [2, 5, 10]:
    n = PolynomialFeatures(d).fit(np.zeros((1, 10))).n_output_features_
    print(f"with 10 input features, degree {d:2d} makes {n:>7,} features")

That last line is the catch: the number of features explodes. On Wednesday you'll see the
**kernel trick**, which gets the benefit of huge feature spaces without ever listing the features.

## Your turn: lift the XOR
In the XOR pattern, opposite corners share a class. No straight line works in 2D.
Which **single** extra feature makes it linearly separable? Add it, and compare the accuracies.

In [ ]:
rng = np.random.default_rng(1)
Xx = rng.uniform(-1, 1, size=(400, 2))
yx = (Xx[:, 0] * Xx[:, 1] > 0).astype(int)          # class 1 in the top-right and bottom-left corners

def lift_xor(X):
    return X                                          # TODO: add one new column

print("2D line:     ", LogisticRegression().fit(Xx, yx).score(Xx, yx))
print("after lifting:", LogisticRegression().fit(lift_xor(Xx), yx).score(lift_xor(Xx), yx))

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
def lift_xor(X):
    return np.c_[X, X[:, 0] * X[:, 1]]                # the product x1*x2 is positive exactly for class 1

print("2D line:     ", LogisticRegression().fit(Xx, yx).score(Xx, yx))
print("after lifting:", round(LogisticRegression(C=100).fit(lift_xor(Xx), yx).score(lift_xor(Xx), yx), 3))
print("One product feature turns a hopeless problem into an easy one: the right feature map is everything.")